<a href="https://colab.research.google.com/github/Se0jwon/Cinema-Note/blob/main/yolo5_maskdetection%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!curl -L https://public.roboflow.com/ds/L90JCi2WZc?key=zJIbM8zJqH > roboflow.zip; unzip roboflow.zip;

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   910  100   910    0     0   2362      0 --:--:-- --:--:-- --:--:--  2357
100 3791k  100 3791k    0     0  4574k      0 --:--:-- --:--:-- --:--:-- 4574k
Archive:  roboflow.zip
 extracting: test/images/w1240-p16x9-0e48e0098f6e832f27d8b581b33bbc72b9967a63_jpg.rf.34ed1e8f70eebdabaf43ab9d40dc1c9b.jpg  
 extracting: test/images/RTX7CCFN_jpg.rf.66ed5c5054f30d933d19ab3d56ace004.jpg  
 extracting: test/images/the-first-day-of-wuhan-s-closure-some-people-fled-some-panicked_jpg.rf.0302fefb0879eb37736a704ca5d070ff.jpg  
 extracting: test/images/1288126-10255706714jpg_jpg.rf.95f7324cbfd48e0386e0660b5e932223.jpg  
 extracting: test/images/126202-untitled-design-13_jpg.rf.56b50d413464989bb2232448a8fbb915.jpg  
 extracting: test/images/r1p00017o8171pnq407_jpg.rf.6fd25b7219a249e97f54fcabf2b52726.jpg  
 extracting: test/images/w1240-p16x9-201

In [2]:
%cd /content/

!git clone https://github.com/ultralytics/yolov5

/content
Cloning into 'yolov5'...
remote: Enumerating objects: 18770, done.
remote: Counting objects: 100% (326/326), done.
remote: Compressing objects: 100% (192/192), done.
remote: Total 18770 (delta 263), reused 134 (delta 134), pack-reused 18444 (from 5)
Receiving objects: 100% (18770/18770), 17.80 MiB | 25.75 MiB/s, done.
Resolving deltas: 100% (12766/12766), done.


In [3]:
!pip install -r yolov5/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 12.8 MB/s eta 0:00:00


In [4]:
%cat data.yaml

train: ../train/images
val: ../valid/images

nc: 2
names: ['mask', 'no-mask']

In [5]:
#클래스 개수를 num_classes에 대입

import yaml

with open("data.yaml", 'r') as stream:

    num_classes = str(yaml.safe_load(stream)['nc'])

In [6]:
# Display the content of models/yolo.py around the problematic import line (line 28) to confirm

print('\n--- Original models/yolo.py content (lines 18-38) ---')

!head -n 38 /content/yolov5/models/yolo.py | tail -n 21


--- Original models/yolo.py content (lines 18-38) ---
import torch
from torch import nn

FILE = Path(__file__).resolve()
ROOT = FILE.parents[1]  # YOLOv5 root directory
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))  # add ROOT to PATH
if platform.system() != "Windows":
    ROOT = Path(os.path.relpath(ROOT, Path.cwd()))  # relative

from models.common import (
    C3,
    C3SPP,
    C3TR,
    SPP,
    SPPF,
    Bottleneck,
    BottleneckCSP,
    C3Ghost,
    C3x,
    Classify,


In [7]:
import sys

import os

import fileinput

In [8]:
def patch_file(filepath, old_line, new_line):

    # Read the file content

    try:

        with open(filepath, 'r') as f:

            lines = f.readlines()

    except FileNotFoundError:

        print(f"Error: File not found at {filepath}")

        return False

    # Modify the content

    modified_lines = []

    modified = False

    for i, line in enumerate(lines):

        if old_line.strip() in line.strip(): # Use 'in' for more flexibility

            # Replace the entire line if it contains the old_line string

            modified_lines.append(new_line + '\n')

            modified = True

            print(f"[{filepath}:{i+1}] Found and replacing: '{line.strip()}' with '{new_line.strip()}'")

        else:

            modified_lines.append(line)



In [9]:
# Patch models/yolo.py to comment out the problematic import

print('\nPatching models/yolo.py...')

%cd /content/yolov5/

patch_file(

    'models/yolo.py',

    'from ultralytics.utils.plotting import feature_visualization',

    '# from ultralytics.utils.plotting import feature_visualization # COMMENTED OUT BY AGENT'

)

# Verify content after patching

print('\n--- Modified models/yolo.py content (lines 18-38) ---')

!head -n 38 /content/yolov5/models/yolo.py | tail -n 21


Patching models/yolo.py...
/content/yolov5

--- Modified models/yolo.py content (lines 18-38) ---
import torch
from torch import nn

FILE = Path(__file__).resolve()
ROOT = FILE.parents[1]  # YOLOv5 root directory
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))  # add ROOT to PATH
if platform.system() != "Windows":
    ROOT = Path(os.path.relpath(ROOT, Path.cwd()))  # relative

from models.common import (
    C3,
    C3SPP,
    C3TR,
    SPP,
    SPPF,
    Bottleneck,
    BottleneckCSP,
    C3Ghost,
    C3x,
    Classify,


In [10]:
# Re-attempt to run training

print('\nAttempting to run train.py after patching models/yolo.py...')

%cd /content/yolov5/

!python train.py --img 416 --batch 16 --epochs 300 --data '../data.yaml' --cfg ./models/yolov5s.yaml --weights '' --name yolov5s_results  --cache


Attempting to run train.py after patching models/yolo.py...
/content/yolov5
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
WARNING ⚠️ wandb: wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
I0000 00:00:1791453520.455819    1757 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: WARNING W&B disabled due to login timeout.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
train: weights=, cfg=./models/yolov5s.yaml, data=../data.yam

In [11]:
!python detect.py --weights runs/train/yolov5s_results/weights/best.pt --img 416 --conf 0.4 --source ../test/images

detect: weights=['runs/train/yolov5s_results/weights/best.pt'], source=../test/images, data=data/coco128.yaml, imgsz=[416, 416], conf_thres=0.4, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-556-gf9578796 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
YOLOv5s summary: 68 layers, 7,015,519 parameters, 0 gradients, 15.8 GFLOPs
image 1/17 /content/test/images/0_Concern-In-China-As-Mystery-Virus-Spreads_jpg.rf.3135dfc5feab288d76a4ccfd22dfc5bf.jpg: 416x416 4 masks, 11.0ms
image 2/17 /content/test/images/1224331650_g_400-w_g_jpg.rf.b816f49e2d84044fc997a8cbd55c347d.jpg: 416x416 4 masks, 11.0ms
image 3/17 /content/test/images/126202-untitled-design

In [12]:
!python detect.py --weights runs/train/yolov5s_results/weights/best.pt --img 416 --conf 0.4 --source ../test/

detect: weights=['runs/train/yolov5s_results/weights/best.pt'], source=../test/, data=data/coco128.yaml, imgsz=[416, 416], conf_thres=0.4, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-556-gf9578796 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
YOLOv5s summary: 68 layers, 7,015,519 parameters, 0 gradients, 15.8 GFLOPs
video 1/1 (1/349) /content/test/마스크영상.mp4: 416x256 1 no-mask, 35.2ms
video 1/1 (2/349) /content/test/마스크영상.mp4: 416x256 1 no-mask, 7.4ms
video 1/1 (3/349) /content/test/마스크영상.mp4: 416x256 1 no-mask, 7.4ms
video 1/1 (4/349) /content/test/마스크영상.mp4: 416x256 1 no-mask, 7.5ms
video 1/1 (5/349) /content/test/마스크영상.mp4: 416x256 1 n